# M5.S3 · Containers, portability and reproducibility in HPC
IE University · Bachelor in Computer Science and Artificial Intelligence · 2026–27 · Official Session 23

Why does a working scientific calculation fail, or change its answer, on another computer?

**Learning outcomes:** diagnose dependency and process-environment differences; distinguish modules, package specifications and containers; record and replay a numerical experiment; explain host/container boundaries; invalidate stale workflow results.

**Five experiments:** E1 dependency drift → E2 process environments → E3 recorded temperature analysis and C/OpenMP → E4 the same payload in Apptainer → E5 workflow cache and independent handover.

Each experiment follows **Real problem → Prediction → Code → Run → Result → Explanation → Student experiment → Takeaway**. Core Python examples are real executions with tiny inputs. Slurm, C/OpenMP and Apptainer results require their own evidence.

**80-minute class:** opening/setup 6 min; E1 8; E2 13; E3 17; E4 17; E5 13; recap/exit 6. Optional reference material is outside this time budget.


## 0 · Run safely on SciTech
1. Open https://jupyter.iesci.tech:8085/; choose Python 3.9+ and save a personal notebook copy.
2. Open via **File → Open from URL** using the branch URL in the README. Run cells in order.
3. Use a **student-owned directory on a filesystem shared with compute nodes**. Inspect [cluster/README.md](../../cluster/README.md), `sinfo` and site modules. Do not SSH directly to worker nodes.
4. Core Python checks take milliseconds and need no installs. The C/OpenMP and optional container checks use a **separate Slurm CPU job**: one node, one task, two CPU slots, 1 GiB, three-minute limit. `RUN_CLUSTER=True` submits this one bounded job if `sbatch` exists; set it False to opt out.
5. `RUN_CONTAINER=False` initially. The teacher supplies an existing trusted, compatible Python SIF; no build, registry pull, package installation or GPU request occurs automatically. Containers do not allocate cluster resources.
6. A Jupyter Slurm job ID is **not evidence that our separate batch job ran**. The notebook collects its own job ID, frozen sources, completion report and case checks.

**Labels:** PASS = executed and checks passed; FAIL = attempted and incorrect/failed; SKIPPED = optional capability disabled/unavailable; NOT RUN = no execution attempted. QUEUED/RUNNING are scheduler states, not numerical success. Expected negative tests report PASS only when the intended rejection was observed. This notebook has no MPI or GPU compute experiment; those integrations are discussed conceptually.


In [ ]:
from pathlib import Path
import os, sys, json, math, random, hashlib, platform, shutil, subprocess, csv, time, venv, uuid, zipfile
ROOT = (Path.cwd() / "m5s3_repro_lab").resolve()
ROOT.mkdir(parents=True, exist_ok=True)
RUN_CLUSTER = True
RUN_CONTAINER = False
IMAGE = ""  # absolute path to a prepared trusted Python 3 SIF, when available
MODULE = "foss/2023b"  # established SciTech toolchain; inspect actual module status in job log
THRESHOLD_C = 25.0     # one-parameter experiment: 20.0 or 22.0
NEW_RUN = False       # True creates a fresh attempt; do not use while your job is queued/running
QUIZ, answers, EVIDENCE = {}, {}, {}
def sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""): h.update(chunk)
    return h.hexdigest()
def call(args, **kw):
    r = subprocess.run([str(x) for x in args], capture_output=True, text=True, timeout=30, **kw)
    if r.returncode: raise RuntimeError(r.stderr or r.stdout or str(r.returncode))
    return r.stdout.strip()
def check(n):
    chosen = str(answers.get(n) or "").strip().upper()
    if chosen not in ("A", "B", "C", "D"):
        print("Choose one letter A–D, then rerun."); return
    print("Correct." if chosen == QUIZ[n]["correct"] else "Review your answer.")
    print(QUIZ[n]["explanation"])
print("LOCAL NOTEBOOK", platform.node(), sys.version.split()[0], sys.executable)
print("Work directory:", ROOT)
print("Slurm submission available:", bool(shutil.which("sbatch")))
print("C/container cases: NOT RUN until a completed batch report is collected.")


## E1 · The same application, a changed dependency
**Real problem:** a temperature conversion changes after a dependency update. We create two tiny local teaching versions of `thermo`; this is not a claim about a published package.

**Prediction:** will identical application text always give the same Kelvin value? Which part of the environment might explain a difference?

**Code / Run:** execute fresh Python processes in the two dependency folders.


In [ ]:
TEMPERATURE_C = 20.0
app = f"import thermo; print(round(thermo.to_kelvin({TEMPERATURE_C}), 3))"
versions = {"v1": "def to_kelvin(x): return x + 273.15\n",
            "v2": "def to_kelvin(x): return (x - 32) * 5 / 9 + 273.15\n"}
observed = {}
for name, source in versions.items():
    folder = ROOT / name
    folder.mkdir(exist_ok=True)
    (folder / "thermo.py").write_text(source)
    observed[name] = call([sys.executable, "-c", app], cwd=folder)
    print(name, "->", observed[name], "K")
assert float(observed["v1"]) == round(TEMPERATURE_C+273.15,3) and observed["v1"] != observed["v2"]
EVIDENCE["E1_dependency_drift"] = "PASS"
print("LOCAL PASS: identical app text, different dependency behaviour.")


**Result / Explanation:** inspect the two actual values. Version 2 uses Fahrenheit semantics. The program can run successfully and still answer the wrong scientific question.

**Student experiment:** change only `TEMPERATURE_C` to 0; predict each answer and rerun. Keep the dependency versions unchanged.

**Takeaway:** record code **and** dependency identity, units and API assumptions.


### Checkpoint Q1
The script and input are unchanged, but a dependency update changes the result. Which evidence helps most?

- **A.** Only the final screenshot
- **B.** Script, input identity and exact dependency environment
- **C.** Only elapsed time
- **D.** Only the input filename


In [ ]:
QUIZ[1] = {'correct': 'B', 'explanation': 'B identifies the changed environment. A loses provenance; C measures duration rather than dependencies; D does not identify input contents or the software.'}
answers[1] = None  # choose A, B, C or D
check(1)


## E2 · Which environment does this process use?
**Real problem:** loading a module in a terminal does not switch an already-running Jupyter kernel.

**Prediction:** can a child process change the parent’s environment? Does a virtual environment include an independent Linux kernel?

**Code / Run:** inspect a real child environment and create an empty real venv without pip or downloads.


In [ ]:
child = "import os; print(os.environ.get('M5S3_SETTING', 'unset'))"
clean = dict(os.environ); clean.pop("M5S3_SETTING", None)
selected = dict(clean, M5S3_SETTING="two-threads")
assert call([sys.executable, "-c", child], env=clean) == "unset"
assert call([sys.executable, "-c", child], env=selected) == "two-threads"
assert os.environ.get("M5S3_SETTING") == clean.get("M5S3_SETTING")
print("PASS: child sees its supplied environment; parent is unchanged.")
EVIDENCE["E2_child_environment"] = "PASS"
env_dir = ROOT / "empty_venv"
try:
    venv.EnvBuilder(with_pip=False).create(env_dir)
    vpy = env_dir / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
    result = json.loads(call([vpy, "-c", "import sys,json; print(json.dumps({'is_venv':sys.prefix!=sys.base_prefix,'python':sys.version.split()[0]}))"]))
    assert result["is_venv"]
    print("PASS: real venv:", result, "(no packages installed)")
    EVIDENCE["E2_venv"] = "PASS"
except (OSError, RuntimeError) as exc:
    print("SKIPPED: venv creation unavailable:", exc)
    EVIDENCE["E2_venv"] = "SKIPPED"


In [ ]:
# Read-only inspection: module is often a shell function, so use bash -lc.
probe = subprocess.run(["bash", "-lc", "if command -v module >/dev/null 2>&1; then module list 2>&1; else echo 'SKIPPED: module command unavailable in this shell'; fi"],
                       capture_output=True, text=True, timeout=15)
print(probe.stdout or probe.stderr)
print("Current kernel still:", sys.executable)
print("This probes the current host; the compute job records its own module status.")


**Result / Explanation:** a venv separates Python package locations while reusing a base interpreter and host system. A module selects site-installed software in a shell. It does not install it into the notebook.

**Spack concept:** `spack.yaml` states requirements; concretization resolves a dependency graph into `spack.lock`. We write an example below; **Spack concretization and installation are NOT RUN**. Preserve the lockfile, Spack version and site/compiler configuration for a real release. A lockfile does not promise universal binary compatibility.

**Student experiment:** change only `M5S3_SETTING` in `selected` to `four-threads`, rerun that child call, and verify the parent stays unchanged.

**Takeaway:** identify the interpreter and environment of the process that actually executes the code.


In [ ]:
spack_yaml = "spack:\n  specs:\n    - zlib\n  concretizer:\n    unify: true\n"
(ROOT / "spack.yaml").write_text(spack_yaml)
print(spack_yaml)
print("NOT RUN: spack concretize / spack install. No lockfile or installation is invented.")


### Checkpoint Q2
A student loads a Python module in a terminal, then returns to an existing notebook kernel. What should they expect?

- **A.** The kernel switches interpreter immediately
- **B.** The module builds a new container
- **C.** The terminal changes; the existing kernel keeps its interpreter
- **D.** The input file changes automatically


In [ ]:
QUIZ[2] = {'correct': 'C', 'explanation': 'C follows process inheritance. A confuses a running process with a future child; B confuses modules with containers; D has no connection to environment selection.'}
answers[2] = None  # choose A, B, C or D
check(2)


## E3 · Record one scientific calculation
**Real problem:** compute the mean temperature and fraction above a threshold from the same 1,000 sensor readings.

**Prediction:** if only the threshold changes, should the input hash or mean change? Should the fraction change?

**Code / Run:** reuse the standard-library `analysis.py` payload from the earlier M5S3 notebook. The source cell also embeds a small C/OpenMP implementation and the batch/replay helpers so no repository checkout is required. The cell only writes sources; it does not run C on the login node. Expand it to inspect the full code.


In [ ]:
SOURCES = {'analysis.py': 'import argparse, csv, json, hashlib, math, platform, sys, os\nfrom pathlib import Path\np = argparse.ArgumentParser()\np.add_argument("--input", required=True)\np.add_argument("--threshold", type=float, default=25.0)\np.add_argument("--output", required=True)\na = p.parse_args()\ninput_path = Path(a.input)\nwith input_path.open(newline="", encoding="utf-8") as f:\n    values = [float(row["temperature_c"]) for row in csv.DictReader(f)]\nif not values:\n    raise ValueError("input must contain temperatures")\nresult = {"count": len(values), "mean_c": math.fsum(values) / len(values),\n          "fraction_above": sum(x > a.threshold for x in values) / len(values)}\ndef sha(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\nrecord = {"result": result, "parameters": {"threshold_c": a.threshold},\n          "input_sha256": sha(input_path), "code_sha256": sha(__file__),\n          "runtime": {"python": sys.version.split()[0],\n                      "platform": platform.system(), "machine": platform.machine(),\n                      "executable": sys.executable,\n                      "omp_threads": os.environ.get("OMP_NUM_THREADS", "unset")}}\nPath(a.output).write_text(json.dumps(record, indent=2) + "\\n", encoding="utf-8")\nprint(json.dumps(result, sort_keys=True))\n', 'temperature.c': '#include <stdio.h>\n#include <stdlib.h>\n#include <math.h>\n#ifdef _OPENMP\n#include <omp.h>\n#endif\nint main(int argc, char **argv) {\n    if (argc != 4) return 2;\n    double threshold = atof(argv[2]), values[2000];\n    int threads = atoi(argv[3]), n = 0, seen = 1;\n    if (threads < 1 || threads > 2) return 2;\n    FILE *f = fopen(argv[1], "r");\n    char line[256];\n    if (!f || !fgets(line, sizeof line, f)) return 3;\n    while (fgets(line, sizeof line, f)) {\n        double x;\n        if (n >= 2000 || sscanf(line, "%lf", &x) != 1 || !isfinite(x)) return 4;\n        values[n++] = x;\n    }\n    fclose(f);\n    if (!n) return 5;\n    double sum = 0;\n    int above = 0;\n#ifdef _OPENMP\n    omp_set_dynamic(0);\n    #pragma omp parallel num_threads(threads) reduction(+:sum,above)\n#endif\n    {\n#ifdef _OPENMP\n        #pragma omp single\n        seen = omp_get_num_threads();\n        #pragma omp for schedule(static)\n#endif\n        for (int i = 0; i < n; i++) {\n            sum += values[i];\n            above += values[i] > threshold;\n        }\n    }\n    if (!isfinite(sum)) return 6;\n    printf("{\\"count\\":%d,\\"mean_c\\":%.17g,\\"fraction_above\\":%.17g,\\"threads\\":%d}\\n",\n           n, sum/n, (double)above/n, seen);\n    return 0;\n}\n', 'worker.py': 'import hashlib, json, math, os, platform, shutil, subprocess, sys\nfrom pathlib import Path\nROOT = Path(__file__).resolve().parent\nos.chdir(ROOT)\ncfg = json.loads((ROOT / "settings.json").read_text())\nreport = {"run_id":cfg["run_id"], "context":"SCITECH_SLURM" if os.environ.get("SLURM_JOB_ID") else "LOCAL_AUTHOR_VALIDATION",\n          "job_id":os.environ.get("SLURM_JOB_ID"), "host":platform.node(), "python":sys.version,\n          "module_status":os.environ.get("M5S3_MODULE_STATUS", "not probed locally"),\n          "cases":{}}\ndef sha(p):\n    h = hashlib.sha256()\n    with Path(p).open("rb") as f:\n        for chunk in iter(lambda: f.read(1024*1024), b""): h.update(chunk)\n    return h.hexdigest()\ndef call(args, **kw):\n    r = subprocess.run([str(x) for x in args], capture_output=True, text=True, timeout=25, **kw)\n    if r.returncode: raise RuntimeError(" ".join(map(str,args))+"\\n"+r.stderr[-3000:]+r.stdout[-1000:])\n    return r.stdout.strip()\ndef agree(a,b):\n    return a["count"] == b["count"] and all(math.isfinite(a[k]) and math.isclose(a[k],b[k],rel_tol=1e-12,abs_tol=1e-12) for k in ("mean_c","fraction_above"))\ndef record(name, status, **kw):\n    report["cases"][name] = dict(status=status, **kw)\n    print(name, json.dumps(report["cases"][name], sort_keys=True), flush=True)\ndef run():\n    for name,h in cfg["source_sha256"].items():\n        if sha(ROOT/name) != h: raise RuntimeError("Frozen input/source changed: "+name)\n    call([sys.executable,"analysis.py","--input","temperatures.csv","--threshold",cfg["threshold_c"],"--output","native.json"])\n    ref=json.loads((ROOT/"native.json").read_text())["result"]\n    expected=cfg["reference"]\n    if not agree(ref,expected): raise RuntimeError("Native Python reference mismatch")\n    record("python", "PASS", result=ref)\n    compiler=shutil.which("gcc")\n    if not compiler:\n        record("c_serial","SKIPPED",reason="gcc unavailable in job")\n        record("c_openmp","SKIPPED",reason="gcc unavailable in job")\n    else:\n        report["compiler_path"]=compiler\n        report["compiler_version"]=call([compiler,"--version"]).splitlines()[0]\n        try:\n            call([compiler,"-std=c11","-O2","-Wall","-Wextra","temperature.c","-lm","-o","temperature_serial"])\n            actual=json.loads(call([ROOT/"temperature_serial","temperatures.csv",cfg["threshold_c"],1]))\n            record("c_serial","PASS" if agree(actual,ref) else "FAIL",result=actual)\n        except Exception as e: record("c_serial","FAIL",reason=str(e))\n        # Probe actual OpenMP compilation separately from scientific correctness.\n        probe=ROOT/"omp_probe.c"\n        probe.write_text("#include <omp.h>\\nint main(void){return omp_get_max_threads()<1;}\\n")\n        r=subprocess.run([compiler,"-fopenmp",str(probe),"-o","omp_probe"],capture_output=True,text=True,timeout=25)\n        if r.returncode:\n            record("c_openmp","SKIPPED",reason="OpenMP compiler/runtime unavailable",diagnostic=r.stderr[-2000:])\n        else:\n            try:\n                call([compiler,"-std=c11","-O2","-Wall","-Wextra","-fopenmp","temperature.c","-lm","-o","temperature_omp"])\n                env=dict(os.environ,OMP_NUM_THREADS="2",OMP_DYNAMIC="FALSE",OMP_PROC_BIND="close",OMP_PLACES="cores")\n                actual=json.loads(call([ROOT/"temperature_omp","temperatures.csv",cfg["threshold_c"],2],env=env))\n                record("c_openmp","PASS" if agree(actual,ref) and actual["threads"]==2 else "FAIL",result=actual)\n            except Exception as e: record("c_openmp","FAIL",reason=str(e))\n    runtime=shutil.which("apptainer")\n    image=Path(cfg["image"]) if cfg["image"] else None\n    if not cfg["run_container"]:\n        record("container","SKIPPED",reason="RUN_CONTAINER=False; native checks still run")\n    elif not runtime:\n        record("container","SKIPPED",reason="Apptainer not visible inside job")\n    elif not image or not image.is_file():\n        record("container","SKIPPED",reason="No readable prepared SIF image; no download attempted")\n    else:\n        try:\n            if sha(image)!=cfg["image_sha256"]: raise RuntimeError("Image hash changed since preparation")\n            # Preserve the exact payload and input; only runtime environment changes.\n            args=[runtime,"exec","--cleanenv","--containall","--bind",str(ROOT)+":/work","--pwd","/work",image,\n                  "python3","analysis.py","--input","temperatures.csv","--threshold",cfg["threshold_c"],"--output","container.json"]\n            call(args)\n            got=json.loads((ROOT/"container.json").read_text())\n            ok=agree(got["result"],ref) and got["input_sha256"]==sha(ROOT/"temperatures.csv") and got["code_sha256"]==sha(ROOT/"analysis.py")\n            record("container","PASS" if ok else "FAIL",result=got["result"],image_sha256=cfg["image_sha256"],runtime_version=call([runtime,"--version"]))\n        except Exception as e: record("container","FAIL",reason=str(e))\ntry:\n    run()\nexcept Exception as e:\n    record("worker","FAIL",reason=repr(e))\nreport["complete"]=True\nreport["status"]="FAIL" if any(x["status"]=="FAIL" for x in report["cases"].values()) else "PASS"\nreport["note"]="Job PASS does not promote SKIPPED capabilities to PASS; inspect each case. No MPI/GPU experiment is performed."\n# Atomic publication: a partial write is never treated as a completed report.\n(ROOT/"report.json.tmp").write_text(json.dumps(report,indent=2)+"\\n")\n(ROOT/"report.json.tmp").replace(ROOT/"report.json")\nprint("M5S3_JOB_DONE status="+report["status"],flush=True)\nraise SystemExit(1 if report["status"]=="FAIL" else 0)\n', 'replay.py': 'import hashlib, json, math, subprocess, sys\nfrom pathlib import Path\nroot = Path(__file__).resolve().parent\nmanifest = json.loads((root / "manifest.json").read_text())\nfor name, expected in manifest["files"].items():\n    actual = hashlib.sha256((root / name).read_bytes()).hexdigest()\n    if actual != expected:\n        print("FAIL: integrity mismatch:", name)\n        raise SystemExit(2)\nresult = subprocess.run([sys.executable, "analysis.py", "--input", "temperatures.csv",\n    "--threshold", str(manifest["threshold_c"]), "--output", "replayed.json"],\n    cwd=root, capture_output=True, text=True, timeout=20)\nif result.returncode:\n    print("FAIL:", result.stderr)\n    raise SystemExit(3)\na = json.loads((root / "replayed.json").read_text())["result"]\nb = json.loads((root / "expected.json").read_text())["result"]\nok = a["count"] == b["count"] and all(math.isclose(a[k], b[k], rel_tol=1e-12, abs_tol=1e-12)\n    for k in ("mean_c", "fraction_above"))\nprint("PASS: independent replay agrees" if ok else "FAIL: numerical mismatch")\nraise SystemExit(0 if ok else 4)\n', 'experiment.def': 'Bootstrap: docker\nFrom: python:3.11-slim\n\n%labels\n    Course HPC_M5S3\n\n%test\n    python3 -c "import csv, json, math"\n\n%runscript\n    exec python3 "$@"\n', 'spack.yaml': 'spack:\n  specs:\n    - zlib\n  concretizer:\n    unify: true\n', 'run.sbatch': '#!/bin/bash -l\n#SBATCH --job-name=m5s3-repro\n#SBATCH --partition=cpu\n#SBATCH --nodes=1\n#SBATCH --ntasks=1\n#SBATCH --cpus-per-task=2\n#SBATCH --mem=1G\n#SBATCH --time=00:03:00\n#SBATCH --output=job-%j.out\n#SBATCH --error=job-%j.err\nset -eo pipefail\ncd "$SLURM_SUBMIT_DIR"\nPYTHON_EXE="$1"\nMPI_MODULE="$2"\nif ! command -v module >/dev/null 2>&1 && [ -r /cvmfs/software.eessi.io/versions/2023.06/init/lmod/bash ]; then\n  source /cvmfs/software.eessi.io/versions/2023.06/init/lmod/bash\nfi\nexport M5S3_MODULE_STATUS="SKIPPED: module command unavailable; probing active compiler"\nif command -v module >/dev/null 2>&1; then\n  if [ -n "$MPI_MODULE" ]; then\n    if module load "$MPI_MODULE"; then export M5S3_MODULE_STATUS="LOADED: $MPI_MODULE";\n    else export M5S3_MODULE_STATUS="FAILED: $MPI_MODULE; probing active compiler"; fi\n  else export M5S3_MODULE_STATUS="UNCHANGED: active site environment"; fi\n  module list 2>&1 || true\nfi\nexport OMP_NUM_THREADS=2 OMP_DYNAMIC=FALSE OMP_PROC_BIND=close OMP_PLACES=cores\n# One task with two CPU slots; every step stays within this resource shape.\nsrun --exact --ntasks=1 --cpus-per-task=2 --cpu-bind=cores "$PYTHON_EXE" worker.py\n'}
for name, source in SOURCES.items():
    (ROOT / name).write_text(source, encoding="utf-8")
print("Materialised:", ", ".join(SOURCES))


In [ ]:
rng = random.Random(23)
input_file = ROOT / "temperatures.csv"
with input_file.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f, lineterminator="\n"); writer.writerow(["temperature_c"])
    writer.writerows([[f"{rng.uniform(10,30):.6f}"] for _ in range(1000)])
def host_run(threshold, output_name):
    output = ROOT / output_name
    print(call([sys.executable, ROOT/"analysis.py", "--input", input_file,
                "--threshold", threshold, "--output", output]))
    return json.loads(output.read_text())
first = host_run(THRESHOLD_C, "host_a.json")
second = host_run(THRESHOLD_C, "host_b.json")
changed = host_run(20.0 if THRESHOLD_C != 20.0 else 25.0, "host_changed.json")
assert first["result"] == second["result"]
assert first["input_sha256"] == changed["input_sha256"]
assert first["result"]["mean_c"] == changed["result"]["mean_c"]
assert first["result"]["fraction_above"] != changed["result"]["fraction_above"]
EVIDENCE["E3_recorded_python"] = "PASS"
print("LOCAL PASS: repeat agrees; a changed threshold changes the fraction only.")
print(json.dumps(first, indent=2))


In [ ]:
# Real computed summary, not an illustrative performance chart.
from html import escape
rows = [("Baseline", THRESHOLD_C, first["result"]),
        ("Changed", changed["parameters"]["threshold_c"], changed["result"])]
print("Run       threshold C    mean C    fraction above")
for name, threshold, r in rows:
    print(f"{name:9} {threshold:11.1f} {r['mean_c']:9.5f} {r['fraction_above']:15.3f}")
svg = ['<svg xmlns="http://www.w3.org/2000/svg" width="720" height="180" viewBox="0 0 720 180">',
       '<rect width="720" height="180" fill="white"/>',
       '<text x="20" y="25" font-family="sans-serif" font-size="18">Computed fraction above threshold (0 to 1)</text>']
for i,(name,threshold,r) in enumerate(rows):
    y = 50+i*55
    svg += [f'<text x="20" y="{y+20}" font-family="sans-serif" font-size="16">{escape(name)} {threshold:g} C</text>',
            f'<rect x="210" y="{y}" width="{450*r["fraction_above"]:.2f}" height="28" fill="{["#356487","#6A8A76"][i]}"/>',
            f'<text x="665" y="{y+20}" font-family="sans-serif" font-size="15">{r["fraction_above"]:.3f}</text>']
svg.append('</svg>'); svg = "\n".join(svg)
(ROOT/"fractions.svg").write_text(svg)
try:
    from IPython.display import SVG, display
    display(SVG(data=svg))
except ImportError:
    print("Plot saved to", ROOT/"fractions.svg", "(IPython display unavailable here)")


### E3a · Inspect the actual C/OpenMP loop
The same CSV, threshold and units feed serial C and two OpenMP threads. Each thread keeps private partial sums, combined by a reduction. Floating-point addition order can differ from Python `math.fsum`; compare with a declared teaching tolerance, not exact JSON bytes.

```c
double sum = 0;
int above = 0;
#pragma omp parallel num_threads(2) reduction(+:sum,above)
{
    #pragma omp for schedule(static)
    for (int i = 0; i < n; i++) {
        sum += values[i];
        above += values[i] > threshold;
    }
}
```
This is a readable excerpt of `temperature.c`; the full program records the actual thread count. It is not a new performance benchmark. The input is deliberately tiny.

**Student experiment:** change only `THRESHOLD_C` to 22, rerun E3 and the preparation/submission/collection below. Predict unchanged mean and input identity, and a changed fraction. Avoid changing image/toolchain at the same time.

**Takeaway:** code + data + parameters + runtime identity make a scientific result explainable.


### Checkpoint Q3
The same input file is analysed at thresholds 25 C and 20 C. Which comparison is expected?

- **A.** The mean must change because the threshold changed
- **B.** The input hash changes automatically
- **C.** Only the fraction above threshold and recorded parameter need change
- **D.** Every byte of the full provenance JSON must match


In [ ]:
QUIZ[3] = {'correct': 'C', 'explanation': 'C describes this calculation. A confuses a statistic with the selection threshold; B confuses a parameter with file contents; D ignores recorded parameters and runtime metadata.'}
answers[3] = None  # choose A, B, C or D
check(3)


## E4 · Run the same payload in an allocated environment
**Real problem:** separate packaging an application from allocating the CPUs that execute it.

**Prediction:** what must stay identical when comparing host and container results? Which host resources does the image not supply?

**Code / Run:** freeze source/input/settings into a unique run directory, submit one bounded CPU job, then collect its report. With `RUN_CONTAINER=False` the real Python and C checks still run; the container case remains SKIPPED. With a prepared image and `RUN_CONTAINER=True`, the same job also runs:

```bash
apptainer exec --cleanenv --containall \
  --bind "$RUN_DIR:/work" --pwd /work "$IMAGE" \
  python3 analysis.py --input temperatures.csv \
  --threshold 25 --output container.json
```
The bind maps shared host files to `/work`. `--containall` reduces default host filesystem visibility for this comparison; it does not make the container independent of the host kernel, runtime or hardware. No image is downloaded. A checksum identifies bytes; it does not establish trust.


In [ ]:
assert math.isfinite(THRESHOLD_C) and -100 <= THRESHOLD_C <= 100
image = Path(IMAGE).expanduser().resolve() if IMAGE else None
image_hash = sha(image) if RUN_CONTAINER and image and image.is_file() else None
frozen = {name: (ROOT/name).read_text() for name in SOURCES}
frozen["temperatures.csv"] = input_file.read_text()
identity = {"source_sha256": {name: hashlib.sha256(text.encode()).hexdigest() for name,text in frozen.items()},
            "threshold_c":THRESHOLD_C, "run_container":RUN_CONTAINER,
            "image":str(image) if image else "", "image_sha256":image_hash, "module":MODULE,
            "python_executable":sys.executable, "reference":first["result"]}
fingerprint = hashlib.sha256(json.dumps(identity,sort_keys=True).encode()).hexdigest()[:20]
RUN_DIR = ROOT / ("run-"+fingerprint+("-"+uuid.uuid4().hex[:6] if NEW_RUN else ""))
RUN_DIR.mkdir(exist_ok=True)
for name, text in frozen.items():
    target = RUN_DIR/name
    if target.exists() and target.read_text() != text: raise RuntimeError("Frozen content mismatch: "+name)
    if not target.exists(): target.write_text(text, encoding="utf-8")
settings = dict(identity, run_id=RUN_DIR.name)
settings_path = RUN_DIR/"settings.json"
if settings_path.exists(): assert json.loads(settings_path.read_text()) == settings
else: settings_path.write_text(json.dumps(settings,indent=2))
print("Frozen run:", RUN_DIR.name)
print("Native C / OpenMP / container: NOT RUN until report is collected.")
print((RUN_DIR/"run.sbatch").read_text())


In [ ]:
JOB_ID = None
state_file = RUN_DIR/"job_id.txt"
if state_file.exists():
    JOB_ID = state_file.read_text().strip()
    if not JOB_ID.isdigit(): raise RuntimeError("Invalid saved job ID")
    print("Reusing job", JOB_ID, "; no duplicate submission.")
elif not RUN_CLUSTER:
    print("NOT RUN: RUN_CLUSTER=False.")
elif not shutil.which("sbatch"):
    print("NOT RUN: sbatch unavailable; local Python evidence is separate.")
else:
    # A different settings fingerprint must not create another job while one is active.
    for previous in ROOT.glob("run-*/job_id.txt"):
        if previous.parent == RUN_DIR: continue
        previous_id = previous.read_text().strip()
        previous_log = previous.parent/("job-"+previous_id+".out")
        if previous_log.exists() and "M5S3_JOB_DONE status=" in previous_log.read_text(errors="replace"): continue
        if not shutil.which("squeue"):
            raise RuntimeError("NOT RUN: verify previous job "+previous_id+" before another submission")
        q = subprocess.run(["squeue","-h","-j",previous_id],capture_output=True,text=True,timeout=10)
        if q.returncode or q.stdout.strip():
            raise RuntimeError("NOT RUN: earlier job is active or unverified: "+previous_id)
    # Avoid inheriting Jupyter's Slurm task/memory settings into a new allocation.
    submit_env = {k:v for k,v in os.environ.items() if not k.startswith("SLURM_")}
    result = subprocess.run(["sbatch", "--parsable", "run.sbatch", sys.executable, MODULE],
                            cwd=RUN_DIR, env=submit_env, capture_output=True, text=True, timeout=25)
    if result.returncode:
        print("FAIL: submission rejected:", result.stderr or result.stdout)
    else:
        JOB_ID = result.stdout.strip().split(";")[0]
        if not JOB_ID.isdigit(): raise RuntimeError("Unexpected sbatch response: "+result.stdout)
        state_file.write_text(JOB_ID)
        print("Submitted CPU job", JOB_ID)
print("Do not resubmit while queued. Rerun the COLLECTION cell below.")


In [ ]:
BATCH_REPORT = None
if not JOB_ID:
    print("NOT RUN: no batch job for this run.")
else:
    report_path = RUN_DIR/"report.json"
    log_path = RUN_DIR/("job-"+JOB_ID+".out")
    err_path = RUN_DIR/("job-"+JOB_ID+".err")
    log = log_path.read_text(errors="replace") if log_path.exists() else ""
    if report_path.exists() and "M5S3_JOB_DONE status=" in log:
        candidate = json.loads(report_path.read_text())
        if not candidate.get("complete") or candidate.get("run_id") != RUN_DIR.name or str(candidate.get("job_id")) != JOB_ID:
            raise RuntimeError("FAIL: report/job identity mismatch")
        BATCH_REPORT = candidate
        print("REAL BATCH REPORT", json.dumps(candidate,indent=2))
    else:
        states = []
        for command in (["squeue","-h","-j",JOB_ID,"-o","%T|%R"],
                        ["sacct","-n","-P","-X","-j",JOB_ID,"-o","State,ExitCode"]):
            if shutil.which(command[0]):
                try:
                    q = subprocess.run(command,capture_output=True,text=True,timeout=10)
                    if q.returncode == 0 and q.stdout.strip(): states.append(q.stdout.strip())
                except (OSError,subprocess.TimeoutExpired): pass
        details = " | ".join(states)
        terminal = any(token in details for token in ("FAILED","CANCELLED","TIMEOUT","OUT_OF_MEMORY","NODE_FAIL","COMPLETED"))
        print(("FAIL: terminal job has no complete evidence. " if terminal else "QUEUED/RUNNING/UNKNOWN: no completed evidence yet. ")+details)
        print("Inspect: squeue -j", JOB_ID, "or sacct -j", JOB_ID)
    if err_path.exists() and err_path.stat().st_size:
        print("Actual stderr:", err_path.read_text(errors="replace")[-2500:])
    if not BATCH_REPORT:
        print("Log tail:", log[-1500:])
        print("Rerun this collection cell later; do not submit again.")


**Result / Explanation:** a completed report checks native Python, serial C and OpenMP separately. Only an actually executed, numerically agreeing container case is PASS. Missing compiler, OpenMP support, runtime or image is reported explicitly. No fallback output masquerades as SciTech evidence.

The default input has 1,000 rows; C and Python may sum in different orders. This teaching comparison uses `rel_tol=abs_tol=1e-12` for the mean/fraction, exact count equality, and actual two-thread verification. Choose tolerances from the scientific method for production work.

**Student experiment:** if an instructor provides a prepared Python SIF, enable only `RUN_CONTAINER` and set its absolute path; compare numerical result, input/code hashes and interpreter metadata. A queued job must finish before you change settings and submit again.

**Takeaway:** Slurm allocates; Apptainer packages; numerical checks establish agreement.

### Optional reference: MPI and GPU boundaries (no execution)
For the Session 22 MPI/OpenMP program use only a site-approved launcher with compatible host/container MPI, PMI/PMIx and interconnect support. Never put an unrelated `mpirun` inside every `srun` task. Existing SciTech notes describe unresolved multi-node library visibility; this notebook does not attempt a multi-node run.

`apptainer exec --nv` exposes NVIDIA devices/host libraries; it does **not** allocate a GPU or guarantee CUDA compatibility. No GPU is needed here, and no GPU or MPI success is claimed.


### Checkpoint Q4
The SIF exists, but the analysis cannot open /work/temperatures.csv. Which check is most useful?

- **A.** Increase the number of GPUs
- **B.** Check the bind source/destination and working directory
- **C.** Rename the image to match the CSV
- **D.** Request more MPI ranks


In [ ]:
QUIZ[4] = {'correct': 'B', 'explanation': 'B tests the path mapping. A and D allocate irrelevant resources; C changes a name without exposing the file.'}
answers[4] = None  # choose A, B, C or D
check(4)


### Checkpoint Q5
The container ran successfully on one CPU node. What follows about a later MPI/GPU run?

- **A.** The image automatically reserves a GPU
- **B.** MPI and GPU compatibility still require their own checks and allocations
- **C.** Two-node MPI is guaranteed to work
- **D.** The runtime will be identical on every system


In [ ]:
QUIZ[5] = {'correct': 'B', 'explanation': 'B separates evidence for each capability. A confuses visibility with allocation; C ignores MPI/launcher dependencies; D ignores hardware, placement and measurement variability.'}
answers[5] = None  # choose A, B, C or D
check(5)


## E5 · Cache correctly and hand over a replayable experiment
**Real problem:** a cached answer can become stale when parameters, code or data change.

**Prediction:** which calls should compute: threshold 25, threshold 25 again, threshold 20?

**Code / Run:** a small teaching cache includes input bytes, code bytes, parameters and runtime identity. This is not a run of Snakemake or Nextflow. Real workflow tools coordinate task dependencies and execution environments; we isolate one correctness rule here.


In [ ]:
cache_file = ROOT/"cache_index.json"
def cached_run(threshold):
    key_data = {"input":sha(input_file), "code":sha(ROOT/"analysis.py"),
                "threshold_c":threshold, "python":sys.version, "executable":sys.executable,
                "system":platform.system(), "machine":platform.machine()}
    key = hashlib.sha256(json.dumps(key_data,sort_keys=True).encode()).hexdigest()
    index = json.loads(cache_file.read_text()) if cache_file.exists() else {}
    output = ROOT/("cached_"+key+".json")
    if key in index and output.exists() and sha(output) == index[key]:
        return "REUSE", json.loads(output.read_text())
    r = host_run(threshold,output.name)
    index[key] = sha(output); cache_file.write_text(json.dumps(index,indent=2))
    return "COMPUTE", r
cache_file.write_text("{}")  # reset only this teaching cache index
cache_results = [cached_run(t) for t in (25,25,20)]
print("Observed actions:", [x[0] for x in cache_results])
assert [x[0] for x in cache_results] == ["COMPUTE","REUSE","COMPUTE"]
EVIDENCE["E5_cache"] = "PASS"
print("LOCAL PASS: parameter change invalidates stale output.")


### E5a · Independent replay and deliberate integrity failure
A second process should reproduce the result without your notebook variables. The handover contains code, input, parameters, an expected result and file identities. A replay first checks file hashes, then numerical agreement. A separate damaged copy must be rejected before scientific calculation.

**Student experiment:** change only threshold 22 in `cached_run(22)`, predict a cache miss, then rerun it unchanged and predict reuse. For integrity, change one input row in the damaged copy only.

**Takeaway:** a reusable environment still needs an explicit workflow, inputs and validation rule.


In [ ]:
handover = ROOT/("handover-"+uuid.uuid4().hex[:8])
handover.mkdir()
for name in ("analysis.py","replay.py","experiment.def","spack.yaml"):
    shutil.copy2(ROOT/name,handover/name)
shutil.copy2(input_file,handover/"temperatures.csv")
# A fresh host record for the selected parameter, independent of cached output.
final = host_run(THRESHOLD_C,"handover_expected.json")
shutil.copy2(ROOT/"handover_expected.json",handover/"expected.json")
manifest = {"threshold_c":THRESHOLD_C,
            "files":{p.name:sha(p) for p in handover.iterdir() if p.is_file()},
            "runtime":final["runtime"], "image_sha256":image_hash if RUN_CONTAINER else None,
            "container_status":BATCH_REPORT["cases"].get("container",{}).get("status","NOT RUN") if BATCH_REPORT else "NOT RUN",
            "comparison":{"rel_tol":1e-12,"abs_tol":1e-12,"note":"Teaching tolerance"}}
(handover/"manifest.json").write_text(json.dumps(manifest,indent=2))
(handover/"README.txt").write_text("Use Python 3.9+ and run: python3 replay.py\nCore replay uses stdlib; venv is recreated, not copied.\nThe definition uses a mutable teaching tag; archive a trusted SIF and its identity for container replay.\nNo MPI/GPU run is included.\n")
print(call([sys.executable,handover/"replay.py"],cwd=handover))
EVIDENCE["E5_independent_replay"] = "PASS"
damaged = ROOT/(handover.name+"-damaged")
shutil.copytree(handover,damaged)
with (damaged/"temperatures.csv").open("a") as f: f.write("99.000000\n")
rejected = subprocess.run([sys.executable,damaged/"replay.py"],cwd=damaged,capture_output=True,text=True,timeout=20)
assert rejected.returncode == 2 and "integrity mismatch" in rejected.stdout
print("LOCAL PASS: deliberate tampering rejected:",rejected.stdout.strip())
EVIDENCE["E5_tamper_rejection"] = "PASS"
archive = ROOT/(handover.name+".zip")
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    for p in handover.iterdir():
        if p.is_file():z.write(p,p.name)
print("Shareable handover:",archive)


### Checkpoint Q6
A workflow’s input filename is unchanged, but the CSV contents or threshold changed. What should happen?

- **A.** Reuse the old result because the filename matches
- **B.** Recompute using input identity and parameters in the cache key
- **C.** Change only the output label
- **D.** Remove provenance so the outputs look identical


In [ ]:
QUIZ[6] = {'correct': 'B', 'explanation': 'B captures dependencies. A overlooks file contents and parameters; C relabels a stale answer; D hides the cause and prevents investigation.'}
answers[6] = None  # choose A, B, C or D
check(6)


## 6 · Four essentials and exit question
1. **Environment:** identify the interpreter, dependencies and compiler that actually ran.
2. **Container:** preserve user-space software while checking host kernel, hardware and runtime integration.
3. **Experiment:** record code, inputs, parameters and a justified numerical comparison.
4. **Workflow:** recompute when dependencies change and hand over a runnable replay.

**Exit question:** a teammate has your SIF but cannot reproduce the result. Name two missing experiment artifacts and one host dependency to check.

**Next session:** Session 24 is the group project workshop. Bring the handover ZIP, source/settings, validation status and one unresolved environment issue. No new theory reading is required for that workshop.


In [ ]:
print("LOCAL PYTHON EVIDENCE (this notebook execution):")
for name, status in EVIDENCE.items(): print(f"{name:30} {status}")
print("SEPARATE BATCH EVIDENCE:")
for name in ("python","c_serial","c_openmp","container"):
    case = BATCH_REPORT["cases"].get(name,{"status":"NOT RUN"}) if BATCH_REPORT else {"status":"NOT RUN"}
    print(f"{name:30} {case['status']}",case.get("reason",""))
print("Spack concretization/install: NOT RUN. MPI/GPU compute: NOT RUN.")
print("Local PASS does not prove SciTech, container or MPI/GPU PASS.")
(ROOT/"my_mcq_answers.json").write_text(json.dumps(answers,indent=2))


## References and validation boundary
**Primary teaching reference:** official 2026–27 syllabus, Session 23, page 16. Main reading: Oscar Diez, *High Performance Computing: Applications and Frontiers*, Chapter 1, modern programming/reproducible science toolchains; supporting reading: Robey and Zamora, *Parallel and High Performance Computing*, Chapter 17. Builds on M3S4’s provenance checks and M5S2’s rank/thread application, then leads into the project workshop.

Official documentation checked 24 October 2026:
- [Modules](https://modules.readthedocs.io/en/stable/module.html)
- [Python venv](https://docs.python.org/3/library/venv.html)
- [Spack environments](https://spack.readthedocs.io/en/latest/environments.html)
- [Apptainer exec](https://apptainer.org/docs/user/main/cli/apptainer_exec.html), [binds](https://apptainer.org/docs/user/main/bind_paths_and_mounts.html), [MPI](https://apptainer.org/docs/user/main/mpi.html), [GPU support](https://apptainer.org/docs/user/main/gpu.html)
- [Slurm sbatch](https://slurm.schedmd.com/sbatch.html)
- [Snakemake](https://snakemake.readthedocs.io/), [Nextflow](https://www.nextflow.io/docs/latest/)

See README and VALIDATION.md for author validation. The published notebook has no prefilled fake SciTech outputs. Local author checks are documented separately; SciTech allocation/module/shared-filesystem and actual Apptainer execution remain to be validated there.
